# Week 11 — Sequence models for user journeys

**Goal.** Find out whether sequence models beat well-engineered tabular features on ads data, and by how much.

**Deliverable.** An LSTM and a small transformer over user timelines, plus an honest tabular-vs-sequence comparison.

**Rough shape of the week.** 2h reading (DIN) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## Set the bar before you build

The interesting result here is quantitative, and it is often *negative*: sequence models
frequently fail to beat a good tabular model with hand-built recency and frequency
features, because those features already capture most of what the sequence contains.

So build the **strong tabular baseline first**: counts, time since last impression, time
since first, campaign diversity, session gaps. Get its number. Only then build the
sequence model. Doing it in that order is what makes the answer credible instead of
self-congratulatory.

Use `split.user_grouped_time_split` — a user must not appear in two folds.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())

seq = (df.sort_values(["uid", "timestamp"])
         .groupby("uid")
         .agg(n=("campaign", "size"), converted=("conversion", "max")))
print(f"{len(seq):,} users, median {seq.n.median():.0f} impressions, "
      f"{(seq.n >= 5).mean():.1%} have >=5")

sp = split.user_grouped_time_split(df, "timestamp", "uid")
print(sp)

## 1. The tabular baseline to beat

Aggregate each user's history into features. Be generous — the point is to make this hard
to beat. Recency, frequency, campaign entropy, inter-arrival statistics, click ratio.

**Leakage warning, and it is subtle:** every aggregate must be computed from events
*strictly before* the impression being scored. Aggregating a user's whole timeline and
attaching it to every row of that timeline uses the future to predict the past, and it
produces a spectacular AUC that means nothing.

In [ ]:
# TODO: expanding-window per-user features, strictly causal

## 2. Sequences

Pad or truncate to a fixed length (start with 20, keep the most recent). Each step: a
campaign embedding plus the time delta since the previous event.

The time delta matters more than people expect — an ads sequence is irregularly sampled,
and a plain LSTM treats "three impressions in one minute" the same as "three impressions
over three weeks". Bucket the log-delta and embed it.

In [ ]:
# TODO: build padded (n_users, seq_len) tensors of campaign ids + time-delta buckets

## 3. LSTM

Packed sequences, so padding does not contribute to the hidden state. Getting that wrong
is quiet: the model still trains, it just learns partly from padding.

In [ ]:
import torch, torch.nn as nn

class SequenceCVR(nn.Module):
    def __init__(self, n_campaigns, n_delta_buckets, dim=32, hidden=64):
        super().__init__()
        # TODO: embeddings -> LSTM -> head; use packed sequences to ignore padding
        raise NotImplementedError

## 4. Transformer, and DIN-style attention

A 2-layer transformer with learned positional embeddings. Then the ads-specific variant
worth more than the vanilla one: **DIN's attention over history, keyed by the candidate
campaign** (`papers/zhou2018-deep-interest-network.pdf`).

DIN's insight is that relevance is not fixed — which parts of a user's history matter
depends on what you are about to show them. That is a genuinely ads-shaped idea rather
than an NLP import, and it is the reason to prefer it here.

In [ ]:
# TODO

## 5. The honest comparison

One table: tabular baseline, LSTM, transformer, DIN. Same users, same split, same metrics.

Then the analysis that makes it publishable rather than a leaderboard:

- **Where does the sequence model win?** Slice by history length. It almost certainly
  adds nothing for single-impression users and everything for long journeys. Plot AUC
  gain against history length.
- **What does it cost?** Training time, inference latency, parameter count. A +0.003 AUC
  for 40x the inference cost is a real answer, and in production it is usually "no".

In [ ]:
# fig, ax = plt.subplots()
# ... AUC gain over tabular baseline, bucketed by user history length
# print(plots.save(fig, 11, "sequence_gain_by_history_length"))

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=11,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=11))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week11_* results/
git commit -m "week 11: <the finding, not the task>"
```